In [1]:
import json

def analyze_sample_leakage(train_path, test_path):
    # 1. 提取训练集所有的实体和上下文
    train_answers = set()
    train_contexts = set()
    with open('/content/drive/MyDrive/数据/Train/Data_train.json', 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            ans = item["answers"]["text"][0].strip().lower()
            ctx = item["context"].strip().lower()
            train_answers.add(ans)
            train_contexts.add(ctx)

    # 2. 统计测试集样本
    total_test = 0
    ans_leaked_count = 0  # 答案在训练集出现过
    ctx_leaked_count = 0  # 句子在训练集出现过
    both_leaked_count = 0 # 句子和答案都出现过（最严重的泄露）

    with open('/content/drive/MyDrive/数据/External/Data_external.json', 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            total_test += 1
            ans = item["answers"]["text"][0].strip().lower()
            ctx = item["context"].strip().lower()

            ans_hit = ans in train_answers
            ctx_hit = ctx in train_contexts

            if ans_hit: ans_leaked_count += 1
            if ctx_hit: ctx_leaked_count += 1
            if ans_hit and ctx_hit: both_leaked_count += 1

    print(f"--- 测试集样本维度泄露分析 ---")
    print(f"测试集总样本数: {total_test}")
    print(f"1. 实体重复样本数: {ans_leaked_count} ({ans_leaked_count/total_test:.2%})")
    print(f"2. 上下文重复样本数: {ctx_leaked_count} ({ctx_leaked_count/total_test:.2%})")


analyze_sample_leakage('train.json', 'test.json')

--- 测试集样本维度泄露分析 ---
测试集总样本数: 529
1. 实体重复样本数: 246 (46.50%)
2. 上下文重复样本数: 0 (0.00%)


In [ ]:
import json
from collections import Counter

train_counter = Counter()
test_counter = Counter()

# train
with open("Data_train.json","r",encoding="utf8") as f:
    for line in f:
        item=json.loads(line)
        ans=item["answers"]["text"][0].strip().lower()
        train_counter[ans]+=1

# test
with open("Data_test.json","r",encoding="utf8") as f:
    for line in f:
        item=json.loads(line)
        ans=item["answers"]["text"][0].strip().lower()
        test_counter[ans]+=1

common=[]

for e in test_counter:
    if e in train_counter:
        common.append((e,train_counter[e],test_counter[e],train_counter[e]+test_counter[e]))

common=sorted(common,key=lambda x:x[3],reverse=True)

print("Top repeated entities")
for x in common[:50]:
    print(x)

Top repeated entities
('sars-cov-2', 279, 68, 347)
('human', 265, 6, 271)
('bioid', 77, 1, 78)
('hek293 cells', 66, 1, 67)
('hek293t cells', 66, 1, 67)
('n', 41, 1, 42)
('zikv', 40, 1, 41)
('ns1', 31, 5, 36)
('a549 cells', 32, 1, 33)
('iav', 29, 1, 30)
('denv', 29, 1, 30)
('24 h', 27, 1, 28)
('kidney', 26, 1, 27)
('capsid', 19, 2, 21)
('hcv', 16, 4, 20)
('nsp3', 14, 2, 16)
('influenza a', 14, 1, 15)
('ebv', 7, 8, 15)
('ap-ms', 12, 2, 14)
('vero e6 cells', 11, 1, 12)
('table 1', 11, 1, 12)
('pb1-f2', 10, 1, 11)
('immunoprecipitation', 9, 1, 10)
('lc-ms/ms', 9, 1, 10)
('nsp2', 8, 1, 9)
('hbv', 7, 2, 9)
('liver', 8, 1, 9)
('ns3', 6, 2, 8)
('72 h', 6, 2, 8)
('sars-cov', 6, 1, 7)
('m2', 6, 1, 7)
('nsp4', 6, 1, 7)
('spike', 5, 1, 6)
('poliovirus', 5, 1, 6)
('mhv', 4, 1, 5)
('co-ip', 4, 1, 5)
('h3n2', 3, 2, 5)
('chikv', 4, 1, 5)
('rack1', 4, 1, 5)
('ptb', 4, 1, 5)
('hbx', 3, 1, 4)
('hek 293t cells', 2, 2, 4)
('huh7.5.1 cells', 1, 3, 4)
('uv cross-linking', 3, 1, 4)
('nucleocapsid (n)', 2, 1, 

In [ ]:
import json

def clean_test_set(train_path, test_path, output_path):
    # 1. 收集训练集中所有的实体答案 (归一化处理)
    train_answers = set()
    with open(train_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            for text in item["answers"]["text"]:
                train_answers.add(text.strip().lower())

    # 2. 过滤测试集
    clean_data = []
    removed_count = 0

    with open(test_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            # 获取当前测试例子的答案
            current_answer = item["answers"]["text"][0].strip().lower()

            # 如果这个答案在训练集里出现过，就跳过（删除）
            if current_answer in train_answers:
                removed_count += 1
                continue

            clean_data.append(item)

    # 3. 保存新的测试集
    with open(output_path, 'w', encoding='utf-8') as f:
        for item in clean_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print(f"清理完成！")
    print(f"从测试集中删除了 {removed_count} 个在训练集中出现过的答案。")
    print(f"新的测试集大小: {len(clean_data)}")

# 执行清理
clean_test_set('train.json', 'test.json', 'test_clean_no_overlap.json')

清理完成！
从测试集中删除了 372 个在训练集中出现过的答案。
新的测试集大小: 162
